# Sun Probability Framework - Comparing 2024 and 2025

**Data**:

- [Sun Probability Framework 2024](https://data.imago.ac.uk/datasets/cloud-probability-statistics-per-small-area-in-2024-version-2-0)
- [Sun Probability Framework 2025](https://data.imago.ac.uk/datasets/cloud-probability-statistics-per-small-area-in-2025-version-2-0)

**Goal**: Compare the 2024 and 2025 Sun Probability Framework (SPF) products to identify how cloud cover conditions changed across the UK.

## Download the data

Download both SPF datasets from the Imago Data Service.

Place both GeoPackages inside the `data/` directory.

The datasets provide annual estimates of annual cloud probability for every small area across the United Kingdom. Therefore, this notebook focuses on the `cloud_probability` variable, which represents the annual average SPF value for these geographies.

## What is Sun Probability Framework?

The Sun Probability Framework (SPF) summarises the probability of cloud cover obscuring the ground. Higher SPF values indicate cloudier conditions and therefore a lower likelihood of direct sunlight reaching the surface, while lower values indicate clearer conditions.


## Installing Libraries

In [ ]:
# Geospatial Datascience libraries
import geopandas as gpd
import numpy as np

# Plotting
from matplotlib.colors import TwoSlopeNorm
import matplotlib.pyplot as plt

## Loading the Datasets

Each dataset contains annual SPF estimates for every small area geography in the United Kingdom.

We will use the `cloud_probability` variable, which contains the annual average SPF.

First, load the two years and check the structure of the file.

In [ ]:

spf_24 = gpd.read_file("data/Cloud probability statistics per small area in 2024 (GeoPackage).gpkg")

spf_25 = gpd.read_file("data/Cloud probability statistics per small area in 2025 (GeoPackage).gpkg")

print(f"SPF 2024: {spf_24.head()}")
print(f"\nSPF 2025: {spf_25.head()}")

### Understanding the Dataset

The principal variables used throughout this notebook are:

- `data_zone_code`: Unique small-area identifier
- `cloud_probability`: Annual average Sun Probability Framework (SPF) value
- `geometry`: small area boundary geometry

For this notebook we only require the annual SPF values.

## Mapping SPF 2025

To have an idea of how SPF looks, it is best if we plot the map of it. For this we will plot `spf_25` as it is the data we are looking at comparing to 2024.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

spf_25.plot(
    column="cloud_probability",
    cmap="Blues",
    legend=True,
    ax=ax,
)
ax.set_axis_off()

fig.suptitle("Sun Probability Framework 2025", fontweight="bold")
fig.show()

The spatial pattern broadly reflects well-known UK climatology. Higher SPF values (greater cloud probability) are generally observed in Scotland, Wales, and the north and west of Britain, where cloud cover tends to be more persistent, while lower SPF values (clearer conditions) are more common across southern and eastern England. This provides useful context before examining how conditions changed between 2024 and 2025.

### Preparing the Data

To compare the two years, we first rename the SPF variable before joining both datasets using their common small-area identifier.

In [ ]:
spf_24 = spf_24.rename(columns={"cloud_probability": "spf_2024"})

spf_25 = spf_25.rename(columns={"cloud_probability": "spf_2025"})

spf = spf_24.merge(
    spf_25[
        [
            "data_zone_code",
            "spf_2025",
        ]
    ],
    on="data_zone_code",
)

spf.head()

## Calculating Annual Change

The difference between the two years provides a simple measure of how cloud cover conditions changed.

Positive values indicate higher SPF (more cloud) in 2025, while negative values indicate lower SPF (less cloud, clearer conditions) in 2025 relative to 2024.

In [ ]:
spf["change"] = spf["spf_2025"] - spf["spf_2024"]

spf["percent_change"] = 100 * spf["change"] / spf["spf_2024"]

## Summary Statistics

Before exploring the spatial patterns, it is useful to summarise the overall national changes.

In [ ]:
print(f"Average SPF 2024: {spf['spf_2024'].mean():.1f}")
print(f"Average SPF 2025: {spf['spf_2025'].mean():.1f}")

print(f"Mean change: {spf['change'].mean():.1f}")
print(f"Median change: {spf['change'].median():.1f}")

print(f"Maximum increase: {spf['change'].max():.1f}")
print(f"Maximum decrease: {spf['change'].min():.1f}")

## Distribution of Annual Change

National averages can hide substantial local variation. A histogram allows us to see whether most small areas experienced similar changes or whether only a small number changed substantially.

### Freedman-Diaconis

Before plotting the distribution, we estimate an appropriate number of histogram bins using the Freedman–Diaconis rule. This method adapts the bin width according to both the number of observations and the variability of the data, helping to reveal the underlying distribution without choosing an arbitrary number of bins.

In [ ]:
# Freedman-Diaconis Calculations
data = spf["change"]
data_points = len(data)
q1 = data.quantile(0.25)
q3 = data.quantile(0.75)

# Calculate interquartile range
iqr = q3 - q1

bin_width = (2 * iqr) / (data_points ** (1 / 3))

spf_change_bins = int(np.ceil((data.max() - data.min()) / bin_width))
spf_change_bins = round(spf_change_bins, 0)
print(f"Amount of SPF bins: {spf_change_bins}")

In [ ]:
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(8, 5))

spf["change"].plot(
    kind="hist",
    color="darkslategrey",
    bins=spf_change_bins,
    density=True,
    alpha=0.6,
    ax=ax,
)

# KDE overlay -- smooths the histogram into a continuous density curve,
# making it easier to see whether the double-peak is a real feature
# or just an artefact of the bin edges
kde = gaussian_kde(spf["change"].dropna())
x_range = np.linspace(spf["change"].min(), spf["change"].max(), 500)
ax.plot(x_range, kde(x_range), color="firebrick", linewidth=2)

ax.set_xlabel("Change in SPF")
ax.set_ylabel("Density")

plt.tight_layout()
plt.title("Distribution of Sun Probability Framework Change, United Kingdom")
plt.show()

In [ ]:
norm = TwoSlopeNorm(
    vmin=spf["change"].min(),
    vcenter=0,
    vmax=spf["change"].max(),
)

fig, ax = plt.subplots(figsize=(8, 10))

spf.plot(
    column="change",
    cmap="RdBu",
    norm=norm,
    legend=True,
    ax=ax,
)

ax.set_title(
    "Change in Sun Probability Framework (2024-2025)",
    fontweight="bold",
)

ax.axis("off")

plt.tight_layout()
plt.show()

## Largest Increases & Decreases

We can identify the small areas experiencing the largest increases and decreases in SPF, allowing us to see specifically which areas are changing the most.

In [ ]:
top = spf.sort_values("change", ascending=False).head(20)
bottom = spf.sort_values("change").head(20)

fig, ax = plt.subplots(figsize=(8, 10))

spf.plot(
    color="lightgrey",
    edgecolor="white",
    linewidth=0.1,
    ax=ax,
)

# Plot as markers at each polygon's centroid
top_centroids = top.geometry.centroid
bottom_centroids = bottom.geometry.centroid

ax.scatter(
    top_centroids.x,
    top_centroids.y,
    color="#2166ac",
    s=60,
    edgecolor="black",
    linewidth=0.5,
    zorder=3,
    label="Largest increase (top 20)",
)

ax.scatter(
    bottom_centroids.x,
    bottom_centroids.y,
    color="#b2182b",
    s=60,
    edgecolor="black",
    linewidth=0.5,
    zorder=3,
    label="Largest decrease (bottom 20)",
)

ax.set_title(
    "Small Areas with the Largest SPF Increases and Decreases (2024-2025)",
    fontweight="bold",
)

ax.axis("off")
ax.legend(loc="lower left", frameon=True, fontsize=9)

plt.tight_layout()
plt.show()

## Overall Direction of Change

In [ ]:
increase = (spf["change"] > 0).sum()
decrease = (spf["change"] < 0).sum()
unchanged = (spf["change"] == 0).sum()

total = len(spf)

print(f"Increased SPF: {increase:,} ({100*increase/total:.1f}%)")
print(f"Decreased SPF: {decrease:,} ({100*decrease/total:.1f}%)")
print(f"No change: {unchanged:,} ({100*unchanged/total:.1f}%)")

# What Have We Learnt?

In this notebook we have:

- introduced the Sun Probability Framework (SPF) product
- explored the spatial distribution of SPF across the United Kingdom
- compared annual SPF values between 2024 and 2025
- quantified national changes using summary statistics
- examined the distribution of annual change using the Freedman–Diaconis rule
- mapped where cloud cover conditions increased and decreased
- identified the small areas experiencing the largest increases and decreases in SPF

This notebook demonstrates how annual SPF products can be used to investigate changes in cloud cover conditions over time. In the next case study, the SPF product will be combined with socioeconomic data to explore how differences in cloud cover conditions relate to broader geographical and social questions.